In [8]:
import pandas as pd

path = "C:/Users/start/Downloads/olist_dataset_csvs/"

dfs = {
    "customers": pd.read_csv(path + "olist_customers_dataset.csv"),
    "orders": pd.read_csv(path + "olist_orders_dataset.csv"),
    "items": pd.read_csv(path + "olist_order_items_dataset.csv"),
    "payments": pd.read_csv(path + "olist_order_payments_dataset.csv"),
    "reviews": pd.read_csv(path + "olist_order_reviews_dataset.csv"),
    "products": pd.read_csv(path + "olist_products_dataset.csv"),
    "sellers": pd.read_csv(path + "olist_sellers_dataset.csv"),
    "geoloc": pd.read_csv(path + "olist_geolocation_dataset.csv"),
    "cat_trans": pd.read_csv(path + "product_category_name_translation.csv"),
}


print("customers:", customers.shape)
print("orders:", orders.shape)
print("items:", items.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("geoloc:", geoloc.shape)
print("cat_trans:", cat_trans.shape)


customers: (99441, 5)
orders: (99441, 8)
items: (112650, 6)
payments: (103886, 5)
reviews: (100000, 7)
products: (32951, 9)
sellers: (3095, 4)
geoloc: (1000163, 5)
cat_trans: (71, 2)


In [9]:
for name, df in dfs.items():
    print("="*70)
    print(name.upper(), "| shape:", df.shape)
    print("columns:", list(df.columns))
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        for c, v in nulls.items():
            print(f"  NULLS -> {c}: {v} ({v/len(df)*100:.2f}%)")
    else:
        print("  no NULLS")
    print("  duplicated rows", df.duplicated().sum())

CUSTOMERS | shape: (99441, 5)
columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
  no NULLS
  duplicated rows 0
ORDERS | shape: (99441, 8)
columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
  NULLS -> order_approved_at: 160 (0.16%)
  NULLS -> order_delivered_carrier_date: 1783 (1.79%)
  NULLS -> order_delivered_customer_date: 2965 (2.98%)
  duplicated rows 0
ITEMS | shape: (112650, 6)
columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'price', 'freight_value']
  no NULLS
  duplicated rows 0
PAYMENTS | shape: (103886, 5)
columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
  no NULLS
  duplicated rows 0
REVIEWS | shape: (100000, 7)
columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment

In [10]:
print("orders: order_id ", orders.order_id.is_unique, "| number of rows:", len(orders))
print("customers: customer_id ", customers.customer_id.is_unique)
print("customers: customer_unique_id ", customers.customer_unique_id.is_unique,
      "| number of unique values:", customers.customer_unique_id.nunique(), "vs total rows:", len(customers))
print("sellers: seller_id ", sellers.seller_id.is_unique)
print("products: product_id ", products.product_id.is_unique)

orders: order_id  True | number of rows: 99441
customers: customer_id  True
customers: customer_unique_id  False | number of unique values: 96096 vs total rows: 99441
sellers: seller_id  True
products: product_id  True


In [12]:
print("--- orders <-> items ---")
print("orders without items:", (~orders.order_id.isin(items.order_id)).sum())
print("number of duplicated order_id in items (means an order has more than one product):", items.order_id.duplicated().sum())
print("maximum number of products in an order:", items.groupby('order_id').size().max())
print("orders with more than one seller (split order):", (items.groupby('order_id')['seller_id'].nunique() > 1).sum())

print()
print("--- orders <-> customers ---")
print("all customer_id in orders exist in customers table?", orders.customer_id.isin(customers.customer_id).all())

print()
print("--- orders <-> reviews ---")
print("orders with reviews:", (~orders.order_id.isin(reviews.order_id)).sum())
print("orders with more than one review:", reviews.order_id.duplicated().sum())

print()
print("--- orders <-> payments ---")
print("orders without payment records:", (~orders.order_id.isin(payments.order_id)).sum())
print("orders with more than one payment:", (payments.order_id.value_counts() > 1).sum())

--- orders <-> items ---
orders without items: 775
number of duplicated order_id in items (means an order has more than one product): 13984
maximum number of products in an order: 21
orders with more than one seller (split order): 1278

--- orders <-> customers ---
all customer_id in orders exist in customers table? True

--- orders <-> reviews ---
orders with reviews: 0
orders with more than one review: 559

--- orders <-> payments ---
orders without payment records: 1
orders with more than one payment: 2961


In [13]:
print("n=", len(customers))
print("number of unique customer_id:", customers['customer_id'].nunique())
print("number of unique customer_unique_id:", customers['customer_unique_id'].nunique())

n= 99441
number of unique customer_id: 99441
number of unique customer_unique_id: 96096


In [14]:
orders_customers = orders.merge(customers[['customer_id','customer_unique_id']], on='customer_id')

purchases_per_customer = orders_customers.groupby('customer_unique_id').size()

total_customers = purchases_per_customer.shape[0]
repeat_customers = (purchases_per_customer > 1).sum()

print("number of real customers:", total_customers)
print("number of customers who made more than one purchase:", repeat_customers)
print("percentage:", round(repeat_customers/total_customers*100, 2), "%")

number of real customers: 96096
number of customers who made more than one purchase: 2997
percentage: 3.12 %


In [15]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

delivered_orders = orders[orders['order_status'] == 'delivered'].copy()
delivered_orders['late_by_days'] = (delivered_orders['order_delivered_customer_date']
                                     - delivered_orders['order_estimated_delivery_date']).dt.days

late_orders = delivered_orders[delivered_orders['late_by_days'] > 0]
print("number of late orders:", len(late_orders))
print("percentage of late orders among delivered orders:", round(len(late_orders)/len(delivered_orders)*100, 2), "%")

number of late orders: 6534
percentage of late orders among delivered orders: 6.77 %


In [16]:
reviews_avg = reviews.groupby('order_id')['review_score'].mean().reset_index()

merged = delivered_orders.merge(reviews_avg, on='order_id', how='left')
merged['is_late'] = merged['late_by_days'] > 0

merged.groupby('is_late')['review_score'].mean()

is_late
False    4.279748
True     2.256581
Name: review_score, dtype: float64

In [17]:
seller_sales = items.groupby('seller_id')['price'].sum().sort_values(ascending=False)

total_sales = seller_sales.sum()
top_10_percent_count = int(len(seller_sales) * 0.10)

top_10_percent_sales = seller_sales.head(top_10_percent_count).sum()

print("عدد البايعين الكلي:", len(seller_sales))
print("عدد أعلى 10% بايعين:", top_10_percent_count)
print("نسبة مبيعاتهم من الإجمالي:", round(top_10_percent_sales/total_sales*100, 2), "%")

عدد البايعين الكلي: 3095
عدد أعلى 10% بايعين: 309
نسبة مبيعاتهم من الإجمالي: 67.49 %
